# W7 · Day 2 — Qdrant Tour: Indices, Metrics, Mini-RAG

**~90 minutes · in-class demo · Jupyter notebook · Track A**

Day 1 you connected to Qdrant and made your first queries. Day 2 goes
deeper: what does HNSW actually do, why does the similarity metric matter,
and how do you assemble a full mini-RAG on top of Qdrant.

Same animals corpus as Day 1 (imported from `wk07_pipeline.py`). We build
on top of it rather than rebuilding.

**Cost per full run:** ~$0.02 (we build a few collections; a bit more OpenAI).

**Notebook flow:**
- Cell 1: Setup + import Day 1's helpers
- Cell 2: Confirm Qdrant reachable + Day 1 collection still exists
- Cells 3-4: The problem indices solve (linear scan timing)
- Cells 5-7: HNSW — Qdrant's default index; how tuning affects recall/speed
- Cells 8-9: Qdrant's quantization mode (IVF's philosophical cousin)
- Cells 10-12: Similarity metrics — cosine vs dot vs L2 on the same corpus
- Cell 13: The silent-bug pattern — non-normalised vectors + wrong metric
- Cells 14-16: Build a mini-RAG on top of Qdrant (5 test questions)
- Cell 17: Wrap + hand-off to Track B

---

## Cell 1 — Setup + import Day 1's pipeline

The helper module `wk07_pipeline.py` contains the same corpus, embedding
functions, and similarity metrics from Day 1. We import them so Day 2
doesn't re-embed everything from scratch.

Open `wk07_pipeline.py` alongside if you want to see what you're importing.

In [ ]:
import os
import time
import numpy as np
from openai import OpenAI

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook"
assert os.environ.get("QDRANT_URL"),     "Set QDRANT_URL — get free-tier at cloud.qdrant.io"
assert os.environ.get("QDRANT_API_KEY"), "Set QDRANT_API_KEY — from your Qdrant Cloud cluster"

from wk07_pipeline import (
    CORPUS_ANIMALS,
    TEST_QUESTIONS,
    embed_batch,
    embed_one,
    cosine, dot, l2,
    get_qdrant_client,
    EMBED_SMALL,
)

openai_client = OpenAI()

print(f"Loaded {len(CORPUS_ANIMALS)} animals from Day 1's helper.")
print(f"Loaded {len(TEST_QUESTIONS)} test questions.")
print("Metrics available: cosine, dot, l2")
print("Pipeline functions: embed_batch, embed_one, get_qdrant_client")

---

## Cell 2 — Reconnect to Qdrant + embed the corpus

Same cluster as Day 1. We'll build several collections today, each
demonstrating a different configuration.

In [ ]:
qdrant = get_qdrant_client()

# Sanity: what collections already exist?
existing = [c.name for c in qdrant.get_collections().collections]
print(f"Existing collections: {existing}\n")

# Embed all 10 animals fresh (Day 1's variables aren't in scope here)
print("Embedding animals with text-embedding-3-small...")
texts = [d["text"] for d in CORPUS_ANIMALS]
vectors = embed_batch(texts, model=EMBED_SMALL)

# Attach to records
for doc, vec in zip(CORPUS_ANIMALS, vectors):
    doc["vector"] = vec

print(f"Embedded {len(vectors)} animals ({len(vectors[0])} dims each).")

---

## Cell 3 — The problem indices solve

Why do vector databases need special indices? Because **linear scan doesn't
scale**. Let's see what a linear scan actually looks like.

**Linear scan** = compute cosine similarity between the query and EVERY
vector in the collection, then sort. That's O(n × d) where n is corpus size
and d is dimension.

At 10 vectors it's instant. At 10 million, it's minutes.

In [ ]:
def linear_scan(query_vec, corpus, k=3):
    """Naive linear scan — compute similarity vs every doc, sort, take top-K."""
    scored = [(cosine(query_vec, doc["vector"]), doc) for doc in corpus]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored[:k]

query = "Which animals live in the ocean?"
q_vec = embed_one(query)

# Time the linear scan on our 10-doc corpus
t0 = time.time()
hits = linear_scan(q_vec, CORPUS_ANIMALS, k=3)
dt = time.time() - t0

print(f"Q: {query!r}")
print(f"Linear scan over {len(CORPUS_ANIMALS)} docs took {dt*1000:.2f} ms\n")
for score, doc in hits:
    print(f"  {score:.3f}  {doc['id']:8s}  ({doc['category']})")

In [ ]:
# Now simulate a bigger corpus to see how linear scan scales.
# We fake it by copying our 10 vectors many times.

print(f"  {'Corpus size':>12s}  {'Linear scan time':>18s}  {'ops (approx)':>15s}")
print(f"  {'-----------':>12s}  {'----------------':>18s}  {'-----------':>15s}")

for size in [10, 100, 1_000, 10_000]:
    # Build a fake corpus by tiling our 10 vectors
    fake_corpus = [{"vector": CORPUS_ANIMALS[i % 10]["vector"],
                    "id": f"fake_{i}"}
                   for i in range(size)]
    
    t0 = time.time()
    scored = [(cosine(q_vec, d["vector"]), d) for d in fake_corpus]
    scored.sort(key=lambda x: x[0], reverse=True)
    _ = scored[:3]
    dt = time.time() - t0
    
    ops = size * 1536
    print(f"  {size:>12,d}  {dt*1000:>15.1f} ms  {ops:>15,d}")

print()
print("At 10K docs, linear scan is still workable (~10s).")
print("At 100K, it's painful. At 10M — completely impractical.")
print("\nSolution: pre-build an *index* that skips most of the work at query time.")

---

## Cell 4 — What indices trade off

**Linear scan is exact.** It compares to every vector, so it always finds the
true top-K. But it's O(n).

**Indexed search is approximate.** It skips most vectors using clever
structure. It might miss the true top-K sometimes, but it's O(log n) or
faster.

The trade-off is **recall vs speed**:
- Perfect recall (100%) = linear scan = slow
- 99% recall = HNSW default = fast
- 95% recall = aggressive index tuning = very fast

In production, 95-99% recall is usually fine — you're already retrieving
top-K, and one occasionally-missed candidate is acceptable.

**Two main index families:**
- **HNSW** — graph-based. Qdrant's default. Fast queries.
- **IVF** — cluster-based. Fast build. Good for very large corpora.

Next cells: HNSW in action.

---

## Cell 5 — HNSW with Qdrant defaults

Create a Qdrant collection using HNSW (the default). Upsert our 10 animals.
Query and observe.

In [ ]:
from qdrant_client.models import Distance, VectorParams, HnswConfigDiff, PointStruct

def recreate_collection(name, size=1536, distance=Distance.COSINE, hnsw=None):
    """Delete-then-create so cells are re-runnable."""
    try:
        qdrant.delete_collection(name)
    except Exception:
        pass
    qdrant.create_collection(
        collection_name=name,
        vectors_config=VectorParams(size=size, distance=distance),
        hnsw_config=hnsw,
    )

def upsert_animals(collection_name):
    """Push the 10 animals into a collection."""
    points = [
        PointStruct(
            id=idx,
            vector=doc["vector"],
            payload={"animal_id": doc["id"], "category": doc["category"], "text": doc["text"]},
        )
        for idx, doc in enumerate(CORPUS_ANIMALS)
    ]
    qdrant.upsert(collection_name=collection_name, points=points)

COLL_HNSW_DEFAULT = "wk07_day2_hnsw_default"
recreate_collection(COLL_HNSW_DEFAULT)  # HNSW is Qdrant's default; no config = defaults
upsert_animals(COLL_HNSW_DEFAULT)

# Show the default HNSW parameters Qdrant used
info = qdrant.get_collection(COLL_HNSW_DEFAULT)
hnsw = info.config.hnsw_config
print(f"Collection {COLL_HNSW_DEFAULT!r} — HNSW default config:")
print(f"  m:               {hnsw.m}          (graph connectivity — higher = more edges per node)")
print(f"  ef_construct:    {hnsw.ef_construct}         (build-time search depth — higher = better recall, slower build)")
print(f"  full_scan_threshold: {hnsw.full_scan_threshold}   (below this many points, use linear scan)")

**Note the `full_scan_threshold`.** With only 10 animals, Qdrant will actually
still do a linear scan — HNSW only kicks in above the threshold. This is
smart: at tiny corpora, linear IS faster than index traversal.

For our demo we're seeing HNSW's CONFIGURATION, not its behaviour. To see
HNSW's behaviour you'd need 10K+ vectors. Trust the mechanism; understand
the parameters.

---

## Cell 6 — HNSW tuned for higher recall

The two knobs that matter for HNSW:
- **`m`** (default 16) — how many graph edges per node. More edges = better
  recall, more memory, slower build.
- **`ef_construct`** (default 100) — how thoroughly the index is built.
  Higher = better recall, slower build.

Build a second collection with aggressive parameters and see how the config
changes.

In [ ]:
COLL_HNSW_TUNED = "wk07_day2_hnsw_tuned"
recreate_collection(
    COLL_HNSW_TUNED,
    hnsw=HnswConfigDiff(m=32, ef_construct=200),  # 2× default on both
)
upsert_animals(COLL_HNSW_TUNED)

info = qdrant.get_collection(COLL_HNSW_TUNED)
hnsw = info.config.hnsw_config
print(f"Collection {COLL_HNSW_TUNED!r} — HNSW tuned config:")
print(f"  m:            {hnsw.m}  (was 16, now 32)")
print(f"  ef_construct: {hnsw.ef_construct}  (was 100, now 200)")
print()
print("Effect at production scale:")
print("  - Better recall (misses fewer true top-K on approximate queries)")
print("  - More memory (each node has 2× the edges to store)")
print("  - Slower build time (index construction does 2× more work)")
print("  - Query speed roughly the same (HNSW query cost is O(log n × ef))")

---

## Cell 7 — Query both collections; results should match

At our tiny scale, both collections should return identical top-3 rankings
(because both fall back to linear scan below `full_scan_threshold`).

At production scale, the tuned collection would have slightly better recall
at the cost of more memory. The pattern to learn: **HNSW parameters give you
a recall-vs-cost lever.**

In [ ]:
query = "Which animals live in the ocean?"
q_vec = embed_one(query)

for coll in [COLL_HNSW_DEFAULT, COLL_HNSW_TUNED]:
    hits = qdrant.query_points(collection_name=coll, query=q_vec, limit=3).points
    print(f"  {coll}:")
    for h in hits:
        p = h.payload
        print(f"    {h.score:.3f}  {p['animal_id']:8s} ({p['category']})")
    print()

**Discussion moment:**
- Did both collections return the same top-3? (They should, at this scale.)
- If they differ, that would indicate HNSW's approximation kicking in.
- **The point of this exercise isn't the results — it's the parameters.** You
  now know Qdrant exposes `m` and `ef_construct` as knobs, and what each
  controls.

---

## Cell 8 — Qdrant quantization (IVF's philosophical cousin)

**IVF** (Inverted File Index) works by clustering vectors first, then only
searching within nearby clusters. Qdrant doesn't ship IVF specifically, but
it ships a related idea: **quantization** — compressing vectors to smaller
representations for faster search.

Both approaches trade some accuracy for a lot of memory + speed.

**Qdrant supports two quantization modes:**
- **Scalar quantization** — each float32 → int8 (4× memory reduction)
- **Product quantization** — vectors split into subvectors and compressed (up to 64× reduction)

We'll enable scalar quantization on a collection and observe the config.

In [ ]:
from qdrant_client.models import ScalarQuantization, ScalarQuantizationConfig, ScalarType

COLL_QUANT = "wk07_day2_quantized"

try:
    qdrant.delete_collection(COLL_QUANT)
except Exception:
    pass

qdrant.create_collection(
    collection_name=COLL_QUANT,
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
    quantization_config=ScalarQuantization(
        scalar=ScalarQuantizationConfig(
            type=ScalarType.INT8,
            quantile=0.99,
            always_ram=True,
        )
    ),
)
upsert_animals(COLL_QUANT)

info = qdrant.get_collection(COLL_QUANT)
print(f"Collection {COLL_QUANT!r} created with scalar quantization.")
print(f"  Storage per vector before:  1536 × 4 bytes = 6144 bytes")
print(f"  Storage per vector after:   1536 × 1 byte  = 1536 bytes  (4× smaller)")
print()
print("On our 10-animal corpus this saves ~45 KB — trivial.")
print("On 10M vectors, this saves 45 GB. That's when quantization pays off.")

---

## Cell 9 — Query the quantized collection

Does quantization change the results? At small scale, usually no — the top-K
structure survives compression. At large scale + tight quantization, results
can shift slightly.

In [ ]:
query = "Which animals live in the ocean?"
q_vec = embed_one(query)

print(f"Q: {query!r}\n")
for coll in [COLL_HNSW_DEFAULT, COLL_QUANT]:
    hits = qdrant.query_points(collection_name=coll, query=q_vec, limit=3).points
    label = "FULL PRECISION" if coll == COLL_HNSW_DEFAULT else "QUANTIZED (int8)"
    print(f"  {label}:")
    for h in hits:
        p = h.payload
        print(f"    {h.score:.3f}  {p['animal_id']:8s} ({p['category']})")
    print()

**Discussion:**
- Same top-3? Same order? Same scores?
- The scores may differ slightly (compression is lossy) but the ranking
  usually holds.
- **Takeaway:** at large scale, quantization gives you 4× memory savings and
  faster queries with usually-negligible quality loss. It's a lever you
  reach for when you outgrow your memory budget.

---

## Cell 10 — Three similarity metrics on the same corpus

Now we shift from indices to metrics. Qdrant supports three:
- **Cosine** — measures angle. Range [-1, 1]. Scale-invariant.
- **Dot product** — measures angle + magnitude. Range depends on vectors.
- **L2 (Euclidean)** — straight-line distance. LOWER is more similar. Range [0, ∞).

We'll create three collections, one per metric, and query them all with the
same question.

In [ ]:
COLL_COSINE = "wk07_day2_metric_cosine"
COLL_DOT    = "wk07_day2_metric_dot"
COLL_L2     = "wk07_day2_metric_l2"

recreate_collection(COLL_COSINE, distance=Distance.COSINE)
recreate_collection(COLL_DOT,    distance=Distance.DOT)
recreate_collection(COLL_L2,     distance=Distance.EUCLID)

upsert_animals(COLL_COSINE)
upsert_animals(COLL_DOT)
upsert_animals(COLL_L2)

print("Created three collections, one per metric. All contain the same 10 animals.")

In [ ]:
query = "What animals hunt from the sky?"
q_vec = embed_one(query)

print(f"Q: {query!r}\n")
for coll, label in [(COLL_COSINE, "COSINE"),
                     (COLL_DOT,    "DOT PRODUCT"),
                     (COLL_L2,     "L2 (Euclidean, lower = closer)")]:
    hits = qdrant.query_points(collection_name=coll, query=q_vec, limit=3).points
    print(f"  {label}:")
    for h in hits:
        p = h.payload
        print(f"    {h.score:>7.3f}  {p['animal_id']:8s} ({p['category']})")
    print()

**Discussion moment:**
- Did all three metrics return the same top-3 animals in the same order?
- The SCORES will differ (each metric produces a different number range).
  Only compare metrics via **rank order**, not raw scores.
- If the rankings agree, that's because OpenAI embeddings are **normalised**
  (unit-length). For unit vectors, cosine = dot product (up to sign), and L2
  is a monotonic function of cosine. So rankings match.
- **What if vectors AREN'T normalised?** Next cell.

---

## Cell 11 — Metric ranges compared

Same pair of vectors, all three metrics. See how the numbers differ.

In [ ]:
vec_cat   = next(d['vector'] for d in CORPUS_ANIMALS if d['id'] == 'cat')
vec_dog   = next(d['vector'] for d in CORPUS_ANIMALS if d['id'] == 'dog')
vec_bee   = next(d['vector'] for d in CORPUS_ANIMALS if d['id'] == 'bee')

pairs = [("cat", "dog", vec_cat, vec_dog, "similar (both mammals)"),
         ("cat", "bee", vec_cat, vec_bee, "different (mammal vs insect)")]

print(f"  {'Pair':15s}  {'cosine':>8s}  {'dot':>8s}  {'l2':>8s}   Interpretation")
print(f"  {'----':15s}  {'------':>8s}  {'---':>8s}  {'--':>8s}   {'-'*30}")
for a, b, va, vb, note in pairs:
    c = cosine(va, vb)
    d = dot(va, vb)
    l = l2(va, vb)
    print(f"  {a:<5s} vs {b:6s}  {c:>8.3f}  {d:>8.3f}  {l:>8.3f}   {note}")

print()
print("Note: for OpenAI embeddings (unit-length), cosine ≈ dot product,")
print("and L2 = sqrt(2 - 2*cosine).")
print("Cosine higher = closer.  Dot higher = closer.  L2 lower = closer.")

---

## Cell 12 — Programme default: cosine. Why?

**Cosine is the safe default for text embeddings.** Reasons:

1. **Scale-invariant.** If you accidentally scale your vectors, cosine still
   works. Dot product doesn't.
2. **Bounded range [-1, 1].** Easy to reason about. L2 has no upper bound.
3. **Matches how embedding models are trained.** Most modern text embedding
   models are trained with cosine as the target metric.
4. **Universally supported.** Every vector DB has it.

**When you'd deviate:**
- **Dot product** — you know your embeddings are normalised AND you want a
  small speedup (one less division per comparison). Marginal at production scale.
- **L2** — you're working with non-text embeddings where magnitude matters
  (e.g. some image or audio embeddings). Rare in RAG.

**Programme rule:** stay on cosine unless you have a specific reason. Log the
reason in your ADR when you deviate.

---

## Cell 13 — The silent-bug pattern

Deck slide 25 warned about this. Let's build it live.

**Scenario:** you use a custom embedding pipeline that doesn't normalise its
output vectors. You configure Qdrant with `dot` distance because someone read
'dot is faster than cosine.' It looks like it works. Your KPIs pass.

But quietly, the rankings are wrong — because dot product favours HIGH-MAGNITUDE
vectors, and non-normalised embeddings vary in magnitude for no meaningful reason.

Let's simulate: take our animal vectors and scale a few of them up artificially.

In [ ]:
# Simulate non-normalised custom embeddings: scale some vectors up 3x for no reason
scaled_docs = []
for d in CORPUS_ANIMALS:
    scale = 3.0 if d['id'] in ["salmon", "ant"] else 1.0   # arbitrarily scaled
    vec = [v * scale for v in d['vector']]
    scaled_docs.append({**d, "vector": vec, "scale": scale})

# Query 'ocean animals' — the honest cosine ranking should put salmon and shark on top.
query = "Which animals live in the ocean?"
q_vec = embed_one(query)

# Rank by cosine (correct — scale-invariant)
cos_scored = [(cosine(q_vec, d['vector']), d) for d in scaled_docs]
cos_scored.sort(key=lambda p: p[0], reverse=True)

# Rank by dot product (WRONG when vectors aren't normalised — magnitude wins)
dot_scored = [(dot(q_vec, d['vector']), d) for d in scaled_docs]
dot_scored.sort(key=lambda p: p[0], reverse=True)

print(f"Q: {query!r}\n")
print("  COSINE ranking (correct):")
for s, d in cos_scored[:5]:
    marker = "  ← scaled 3x" if d['scale'] > 1 else ""
    print(f"    {s:>7.3f}  {d['id']:8s} ({d['category']:7s}){marker}")

print("\n  DOT PRODUCT ranking (buggy — magnitude wins over meaning):")
for s, d in dot_scored[:5]:
    marker = "  ← scaled 3x" if d['scale'] > 1 else ""
    print(f"    {s:>7.3f}  {d['id']:8s} ({d['category']:7s}){marker}")

**This is the silent bug.**
- Cosine returns salmon + shark at the top (correct — they're the actual ocean animals)
- Dot product returns salmon + ant at the top (wrong — ant scored high just because we scaled its vector)
- **In production this would look fine on some queries and quietly break others.**

**Prevention:**
1. **Always normalise your embedding vectors** if you're going to use dot product.
2. **Default to cosine** so this class of bug can't happen.
3. **Add a sanity check** that a vector's L2 norm is ~1.0 before upserting.

**Programme rule:** cosine. Unless you have a specific reason and have run
this exact test. Deck slide 26's takeaway.

---

## Cell 14 — Mini-RAG on top of Qdrant

Now assemble a full mini-RAG. Same shape as W6's `ask_rag()`, but Qdrant is
the storage. We'll use the cosine collection from Cell 10.

This mirrors exactly what you'll do in Track B for your capstone.

In [ ]:
SYSTEM = (
    "You are a helpful assistant. Answer using ONLY the provided context. "
    "If the context does not contain the answer, say so plainly. Cite the "
    "animal id in square brackets after any fact you use."
)

def ask_qdrant_rag(question: str, collection: str, k: int = 3) -> dict:
    """Full mini-RAG: embed → Qdrant retrieve → prompt → generate."""
    # 1. Embed the query
    q_vec = embed_one(question)
    
    # 2. Retrieve top-K from Qdrant
    hits = qdrant.query_points(
        collection_name=collection,
        query=q_vec,
        limit=k,
    ).points
    
    # 3. Build the prompt with retrieved chunks
    context = "\n\n".join(
        f"[{h.payload['animal_id']}]\n{h.payload['text']}"
        for h in hits
    )
    
    # 4. Generate
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content":
                f"Context:\n{context}\n\n---\n\nQuestion: {question}"},
        ],
    )
    
    return {
        "question": question,
        "answer":   resp.choices[0].message.content,
        "sources":  [h.payload["animal_id"] for h in hits],
    }

# Test on one question
result = ask_qdrant_rag("Which animals live in the ocean?", COLL_COSINE)
print(f"Q: {result['question']}\n")
print(f"A: {result['answer']}\n")
print(f"Sources: {result['sources']}")

---

## Cell 15 — Run all 5 test questions

Now the full test set. Watch how well the mini-RAG handles each.

In [ ]:
for tq in TEST_QUESTIONS:
    result = ask_qdrant_rag(tq["q"], COLL_COSINE, k=3)
    print(f"── {tq['q']}")
    print(f"   expected categories: {tq['expected_categories']}")
    print(f"   sources retrieved:   {result['sources']}")
    print(f"   answer: {result['answer']}")
    print()

**Discussion:**
- Did every question retrieve at least one animal from the expected category?
- Which question was hardest?
- Did the LLM correctly cite `[animal_id]` for each fact it used?
- Look at Question 4 ("Which animals are kept as pets?") — did cat and dog
  come back? Did gecko?

---

## Cell 16 — Cleanup

Delete the collections we created today so your Qdrant cluster stays tidy.

(In your capstone Track B work you'll create ONE persistent collection —
`capstone_chunks`. These demo collections are transient.)

In [ ]:
for coll in [COLL_HNSW_DEFAULT, COLL_HNSW_TUNED, COLL_QUANT,
             COLL_COSINE, COLL_DOT, COLL_L2]:
    try:
        qdrant.delete_collection(coll)
        print(f"  deleted {coll}")
    except Exception as e:
        print(f"  skip {coll}: {e}")

print("\nDay 1's animals collection (wk07_day1_animals) is preserved.")
print("Delete manually if you don't want it.")

---

## Cell 17 — Wrap: what you learned across two days

**Day 1 — Embedding space:**
1. Compared `3-small` vs `3-large`. 6.5× cost doesn't buy 6.5× quality.
2. Built intuition for semantic geometry — categories cluster.
3. Toured 4 vector DBs. Programme default: Qdrant.
4. Made your first Qdrant call.

**Day 2 — Qdrant tour:**
5. **Indices** exist because linear scan doesn't scale beyond ~10K docs.
6. **HNSW** is Qdrant's default. Two knobs: `m` (connectivity) and
   `ef_construct` (build depth). Both trade recall vs cost.
7. **Quantization** compresses vectors (int8 = 4× smaller). Pays off at scale.
8. **Cosine, dot, L2** — three distance metrics. Cosine is the safe default.
9. **The silent-bug pattern** — non-normalised vectors + dot product = wrong
   rankings that look fine. Stay on cosine.
10. **Mini-RAG on Qdrant** — same shape as W6's ask_rag, different storage.

**Track B (take-home): migrate YOUR capstone.**

Open `AI-RAG_W7_Application_Growth_Guide.md` and follow the steps. Same
pattern you just did with 10 animals, applied to your capstone's 100-300
real chunks. About 2 hours self-paced.

You will:
- Add `src/rag/qdrant_store.py` (~60 lines)
- Add `src/rag/qdrant_rag.py` (~80 lines) — parallel to W6's naive_rag
- Add `scripts/migrate_to_qdrant.py` — one-off migration
- Swap one function call in `src/api/main.py`
- Update `docs/adr/0001-capstone-framing.md` with vector-stack decisions
- Fill `docs/kpi/wk7-snapshot.md`